Requirements: selenium

In [ ]:
# imports
import selenium, time
from selenium import webdriver

Note: Every kayak.com search URL is built like this:

- /flights
- /ORIGIN-DESTINATION
- /YYYY-MM-DD

The part followed after the date will be generated automatically: `?ucs=13qp7li&sort=bestflight_a`

In [ ]:
# Create a new driver and scrape an example KAYAK search
driver = webdriver.Chrome()
driver.get("https://www.kayak.com/flights/ZRH-BER/2026-05-13?ucs=13qp7li&sort=bestflight_a")
time.sleep(5)

### Structure of KAYAK and its flights

Every flight listened on the search is inside a `<div>` with the following class name: `Fxw9-result-item-container`.

While the `Fxw9` part is dynamically changed, `-result-item-container` is always the same.

Ideally, we want to go for this div's via Regex or Xpath.

In [ ]:
flights = driver.find_elements('xpath', "//div[contains(@class, '-result-item-container')]")
len(flights)

# I just noticed that Ad's are also inside the same class. Tho we can filter them out because they have a div containing the text "Ad".

flights_cleaned = []

for f in flights:
    is_ad = f.find_elements('xpath', './/div[text()="Ad"]')
    if not is_ad:
        flights_cleaned.append(f)

print(len(flights_cleaned))

### Define flight informations we need

Ideally we want to keep the following informations about a flight:

- Departure/Arrival time
- Stopover?
- Origin/Destination
- Flight time
- Baggage included?

For that we have to analyse the div with all the flights we obtained.

After doing so, here are the takeaways:

- **Price**: Is inside a `div` with class name: `e2GB-price-text`
- **Operator**: Is inside a `div` with attribute `dir` named `ltr`
- **Stopover**: Is inside a `div` with class name: `vmXl vmXl-mod-variant-default`
- **Flight time**: Is inside a `div` with class name: `vmXl vmXl-mod-variant-large`
- **Origin/Destination**: We don't need to scrape this information as were providing it ourselfes as parameter for flight search.
- **Baggage**: Is a bit more complicated - will be looking for it after acquiring the most basic informations mentioned above.

In [ ]:
# We have all flights already in our `flights` variable. We will start searching from there for the prices first.
prices = []

for flight in flights:
    try:
        price_element = flight.find_element('xpath', ".//div[contains(@class, '-price-text')]")
        prices.append(price_element.text)
    except:
        continue 

We could extract all the prices by going through our div we found earlier. We are using a try catch to skip anything else which does not contain `-price-text` in its class name to prevent the script from crashing. We can now do the same for the other informations.

In [ ]:
stopovers = []
flight_times = []
operators = []

for flight in flights:
    try:
        stopover = flight.find_element('xpath', ".//div[contains(@class, '-mod-variant-default')]/span")
        stopovers.append(stopover.text)
        flight_time = flight.find_element('xpath', ".//div[contains(@class, '-mod-variant-large')]")
        flight_times.append(flight_time.text)
        operator = flight.find_element('xpath', "//div[contains(@dir, 'ltr')]")
        operators.append(operator.text)
    except:
        continue

### Is our information correct?

We gathered most of the info besides the baggages now. We have to check if all lists have the same length. This is important, otherwise we can't zip them together and the data would not match together and be considered as "false information".

In [ ]:
print(len(flights))
print(len(flight_times))
print(len(stopovers))
print(len(operators))
print(len(prices))

Should be correct, as we have on every information we need the same amount. We could start building a flights list:

In [ ]:
all_flights = zip(operators, flight_times, stopovers, prices)

In [ ]:
print("Flights from ORI to DES on 2026-05-23:\n")

for o, f, s, p in all_flights:
    print(f"{o}\n{f}\n{s}\n{p}\n")
